# Human-mouse leaderboard

Regenerates the leaderboard directly from `outputs/` (metrics recomputed from
predictions + GT) with cost, cost breakdown, failure/retry rates and retry-error info.
Re-run after any change under `outputs/` to refresh.

In [ ]:
# ruff: noqa: T201, E402
import os
from pathlib import Path

while Path.cwd().name in ("evaluation", "inference", "preprocessing", "research", "notebooks"):
    os.chdir("..")

import pandas as pd

from config.config import METRICS_DIR
from src.leaderboard import build_leaderboard, rank_leaderboard, retry_error_summary
from src.plots import scatter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

## Build the full board (one row per experiment x prompt)

In [ ]:
SUBSET = "human-mouse"
WITH_COST = True   # set False to skip the OpenRouter pricing lookup (offline)

board = build_leaderboard(subset=SUBSET, with_cost=WITH_COST)
print(f"{len(board)} rows | {board['model'].nunique()} models | "
      f"experiment types: {sorted(board['experiment_type'].unique())}")
board.head()

In [ ]:
# Persist so the leaderboard can be diffed / shared
save_path = METRICS_DIR / "estimations" / f"{SUBSET}_leaderboard.csv"
save_path.parent.mkdir(parents=True, exist_ok=True)
board.to_csv(save_path, index=False)
print(f"Saved {len(board)} rows to {save_path}")

## Ranked leaderboard — reliable runs (fail rate <= 5%), best config per model

In [ ]:
ranked = rank_leaderboard(
    board[board["experiment_type"] == "models-estimation"],
    metric="F1 Score", best_per_model=True, max_fail_rate=0.05,
)
metric_cols = ["model", "prompt", "F1 Score", "Youden's index", "Accuracy",
               "Precision", "Recall", "fail_rate", "mean_num_retries"]
ranked[metric_cols].head(15)

## Cost report — total and breakdown (input / cached / output / reasoning)

In [ ]:
cost_cols = ["model", "prompt", "F1 Score", "cost_total_$", "cost_input_$",
             "cost_cached_$", "cost_output_$", "cost_reasoning_$"]
ranked[[c for c in cost_cols if c in ranked.columns]].head(15)

## Reliability report — failure rate, retry rate and retry-error breakdown

In [ ]:
retry_error_summary(board[board["experiment_type"] == "models-estimation"]).head(20)

## F1 vs cost (reliable models-estimation runs)

In [ ]:
plot_df = ranked.dropna(subset=["cost_total_$", "F1 Score"])
scatter(plot_df, x="cost_total_$", y="F1 Score", color="model", hover_data=["prompt"],
        title=f"F1 vs total cost per run ({SUBSET}, reliable runs)",
        labels={"cost_total_$": "Total cost per run (USD)", "F1 Score": "F1 Score"},
        plot_kwargs={"width": 950, "height": 550})